# 하남교산 예측위험도 770격자 재생성

`02._격자_(하남교산).geojson` (770개)를 기준으로 `23._토지이용계획도_(하남교산).geojson`과 공간조인하여
격자당 1행, 총 770행의 예측위험도를 산출합니다.

**입력 파일**
- `data/02._격자_(하남교산).geojson`
- `data/23._토지이용계획도_(하남교산).geojson`
- `epdo_analysis/output/08_격자_종합위험지수.csv`

**출력:** `epdo_analysis/output/13_하남교산_예측위험도.csv`

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import geopandas as gpd

## 경로 및 상수 설정

In [ ]:
BASE_DIR   = Path(".").resolve()
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "epdo_analysis" / "output"

INPUT_GRID      = DATA_DIR   / "02._격자_(하남교산).geojson"
INPUT_LU        = DATA_DIR   / "23._토지이용계획도_(하남교산).geojson"
INPUT_GRID_RISK = OUTPUT_DIR / "08_격자_종합위험지수.csv"
OUT_PATH        = OUTPUT_DIR / "13_하남교산_예측위험도.csv"

CRS_PROJ = "EPSG:5186"

SCHOOL_TYPES     = {"학교", "초등학교", "중학교", "고등학교", "교육시설"}
RESIDENT_TYPES   = {"아파트", "연립주택", "다세대주택", "단독주택", "공동주택", "주상복합", "연립"}
COMMERCIAL_TYPES = {"상업", "일반상업", "근린상업", "근린생활시설용지", "상업시설"}

for name, p in [
    ("격자",         INPUT_GRID),
    ("토지이용",     INPUT_LU),
    ("종합위험지수", INPUT_GRID_RISK),
]:
    print(f"{name:10s}: {'OK' if p.exists() else 'NOT FOUND'}  {p}")

## Step 1 · blockType별 평균 위험지수 산출

In [ ]:
lu_risk = defaultdict(list)
with open(INPUT_GRID_RISK, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        land_use = (row.get("land_use") or "").strip()
        ecr      = row.get("entropy_composite_risk")
        if land_use and land_use != "미분류" and ecr:
            try:
                lu_risk[land_use].append(float(ecr))
            except ValueError:
                pass

lu_avg_risk = {lt: round(sum(v) / len(v), 2) for lt, v in lu_risk.items()}
all_risks   = [r for vals in lu_risk.values() for r in vals]
overall_avg = round(sum(all_risks) / len(all_risks), 2) if all_risks else 0.0

lu_vals = sorted(lu_avg_risk.values())
def _pct(vals, p):
    if not vals:
        return overall_avg
    idx = (p / 100) * (len(vals) - 1)
    lo, hi = int(idx), min(int(idx) + 1, len(vals) - 1)
    return vals[lo] + (idx - lo) * (vals[hi] - vals[lo])

grade_high = _pct(lu_vals, 75)
grade_mid  = _pct(lu_vals, 25)

print(f"토지이용 유형 수: {len(lu_avg_risk)}")
print(f"전체 평균 위험지수: {overall_avg}")
print(f"고위험 기준(75%): {grade_high:.2f} / 중위험 기준(25%): {grade_mid:.2f}")

## Step 2 · 하남교산 격자 및 토지이용 로드

In [ ]:
grid_gdf = gpd.read_file(INPUT_GRID).to_crs(CRS_PROJ)
gid_col  = next(
    (c for c in grid_gdf.columns if c.lower() in ("gid", "grid_gid", "grid_id")),
    grid_gdf.columns[0],
)
grid_gdf = grid_gdf.rename(columns={gid_col: "gid"})
grid_gdf = grid_gdf[["gid", "geometry"]]

lu_gdf = gpd.read_file(INPUT_LU).to_crs(CRS_PROJ)
lu_gdf = lu_gdf[["blockType", "geometry"]]

print(f"격자 수     : {len(grid_gdf):,}")
print(f"토지이용 수 : {len(lu_gdf):,}")

## Step 3 · 공간 조인 (격자 × 토지이용)

In [ ]:
joined = gpd.sjoin(grid_gdf, lu_gdf, how="left", predicate="intersects")
joined = joined.drop_duplicates(subset="gid", keep="first")
joined["blockType"] = joined["blockType"].fillna("미분류").astype(str).str.strip()
joined = joined.drop(columns=["index_right"], errors="ignore")

print(f"조인 결과 행 수: {len(joined):,}  (목표: 770)")
joined[["gid", "blockType"]].head()

## Step 4 · 격자별 pred_risk, risk_grade 산출

In [ ]:
result = []
for _, row in joined.iterrows():
    btype = row["blockType"]
    if btype in lu_avg_risk:
        pred_risk, basis = lu_avg_risk[btype], "동일유형"
    elif btype in SCHOOL_TYPES:
        matched   = [lu_avg_risk[k] for k in SCHOOL_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis     = "학교유형평균"
    elif btype in RESIDENT_TYPES:
        matched   = [lu_avg_risk[k] for k in RESIDENT_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis     = "주거유형평균"
    elif btype in COMMERCIAL_TYPES:
        matched   = [lu_avg_risk[k] for k in COMMERCIAL_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis     = "상업유형평균"
    else:
        pred_risk, basis = overall_avg, "전체평균"

    risk_grade = "고위험" if pred_risk >= grade_high else ("중위험" if pred_risk >= grade_mid else "저위험")
    result.append({
        "grid_gid"  : row["gid"],
        "blockType" : btype,
        "pred_risk" : pred_risk,
        "risk_grade": risk_grade,
        "basis"     : basis,
    })

result.sort(key=lambda x: -x["pred_risk"])

high = sum(1 for r in result if r["risk_grade"] == "고위험")
mid  = sum(1 for r in result if r["risk_grade"] == "중위험")
low  = sum(1 for r in result if r["risk_grade"] == "저위험")
print(f"격자 수: {len(result)}개")
print(f"고위험: {high} | 중위험: {mid} | 저위험: {low}")

## 결과 저장

In [ ]:
cols = ["grid_gid", "blockType", "pred_risk", "risk_grade", "basis"]
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"저장: {OUT_PATH}")
print(f"격자 수: {len(result)}개 (770개 목표)")